# Split Cleaned Reviews into Positive and Negative Files

This notebook splits the cleaned CSV from the Data Preparation notebook into separate files by sentiment label.

The notebook **auto-detects** the label column and the positive/negative label values. You only need to set your cleaned CSV file name below.

**Output:** Two CSV files (one per sentiment class), used by the **LDA Topic Modeling POS** and **LDA Topic Modeling NEG** notebooks. By splitting reviews by sentiment before topic modeling, we can discover what themes appear specifically in positive vs. negative feedback.

**Run this notebook after** the Data Preparation notebook has produced the cleaned CSV.

In [ ]:
import pandas as pd

# ============================================================
# CHANGE THIS VALUE FOR YOUR DATASET
# ============================================================

input_file = "coffee_maker_binary_cleaned.csv"  # Cleaned CSV from the Data Preparation notebook

# ============================================================
# DO NOT CHANGE ANYTHING BELOW THIS LINE
# ============================================================

df = pd.read_csv(input_file)
print(f"Loaded: {len(df)} rows")
print(f"Columns: {list(df.columns)}\n")

# ---- Auto-detect the label column ----
# Look for a non-numeric, non-text column with exactly 2 unique values.
# Prefer columns whose name contains common label keywords.
# Exclude 'cleaned_text' and any column with many unique values (likely text).
label_keywords = ["sentiment", "label", "class", "category", "target"]
candidates = []
for col in df.columns:
    if col in ["cleaned_text", "combined_text", "text", "review"]:
        continue
    n_unique = df[col].nunique()
    if n_unique == 2:
        name_match = any(kw in col.lower() for kw in label_keywords)
        candidates.append((col, name_match))

if not candidates:
    print("ERROR: Could not auto-detect a label column with exactly 2 unique values.")
    print(f"Available columns and their unique counts:")
    for col in df.columns:
        print(f"  {col}: {df[col].nunique()} unique values")
    raise ValueError("Label column not found. Check your dataset.")

# Prefer keyword matches
candidates.sort(key=lambda x: not x[1])
label_column = candidates[0][0]
print(f"Auto-detected label column: '{label_column}'")

# ---- Auto-detect positive and negative labels ----
# Use simple heuristics: look for known positive/negative keywords.
# If neither label matches keywords, assign based on class size
# (larger class = positive, smaller class = negative).
labels = df[label_column].unique().tolist()
print(f"Label values found: {labels}\n")

positive_keywords = ["positive", "pos", "good", "yes", "1", "true", "like", "high"]
negative_keywords = ["negative", "neg", "bad", "no", "0", "false", "dislike", "low"]

pos_label = None
neg_label = None

for label in labels:
    label_lower = str(label).lower().strip()
    if any(kw == label_lower for kw in positive_keywords):
        pos_label = label
    elif any(kw == label_lower for kw in negative_keywords):
        neg_label = label

# If keyword matching didn't resolve both, use class size
if pos_label is None or neg_label is None:
    counts = df[label_column].value_counts()
    sorted_labels = counts.index.tolist()
    if pos_label is None and neg_label is None:
        # Larger class = positive, smaller class = negative
        pos_label = sorted_labels[0]
        neg_label = sorted_labels[1]
        print(f"No keyword match found. Assigning by class size:")
    elif pos_label is None:
        pos_label = [l for l in labels if l != neg_label][0]
    else:
        neg_label = [l for l in labels if l != pos_label][0]

print(f"Positive label: '{pos_label}' ({len(df[df[label_column] == pos_label]):,} rows)")
print(f"Negative label: '{neg_label}' ({len(df[df[label_column] == neg_label]):,} rows)")

# ---- Split by sentiment ----
pos_df = df[df[label_column] == pos_label].copy()
neg_df = df[df[label_column] == neg_label].copy()

# Auto-generate output filenames from input filename
base_name = input_file.replace("_cleaned.csv", "").replace(".csv", "")
pos_file = f"{base_name}_cleaned_POS.csv"
neg_file = f"{base_name}_cleaned_NEG.csv"

pos_df.to_csv(pos_file, index=False)
neg_df.to_csv(neg_file, index=False)

print(f"\nSaved: {pos_file} ({len(pos_df):,} rows)")
print(f"Saved: {neg_file} ({len(neg_df):,} rows)")
print(f"\nCopy these files to the NLP directory for the LDA notebooks.")